In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 백터DB:Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

# 1. knowlege base 구성을 위한 데이터 생성

In [22]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 4.12 s
Wall time: 4.22 s


180

In [10]:
# 임베딩 객체
# from dotenv import load_dotenv
# from langchain_upstage import UpstageEmbeddings
# load_dotenv()
# embedding = UpstageEmbeddings(
#     #api_key = os.getenv('OPENAI_API_KEY'),
#     model = 'solar-embedding-1-large-passage'
# )

In [2]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'text-embedding-3-large'
)

In [3]:
len(embedding.embed_query("샘플"))

3072

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [4]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    #api_key=os.getenv(PINECONE_API_KEY)
)
# 데이터를 처음 저장할때
index_name = 'tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )

# 업로드된 백터DB를 load할때
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 578 ms
Wall time: 1.99 s


# 2. 답변생성을 위한 Retrieval

In [5]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query,k=3)


In [8]:
retrieved_doc="\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])
print(retrieved_doc)

[전문개정 2009. 12. 31.]  제10조(납세지의 변경신고) 거주자나 비거주자는 제6조부터 제9조까지의 규정에 따른 납세지가 변경된 경우 변경된 날부터 15일 이내에 대통령령으로 정하는 바에 따라 그 변경 후의 납세지 관할 세무서장에게 신고하여야 한다. [전문개정 2009. 12. 31.]  제11조(과세 관할) 소득세는 제6조부터 제10조까지의 규정에 따른 납세지를 관할하는 세무서장 또는 지방국세청장이 과세한다. [전문개정 2009. 12. 31.]  제2장 거주자의 종합소득 및 퇴직소득에 대한 납세의무 <개정 2009. 12. 31.>  제1절 비과세 <개정 2009. 12. 31.>  제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.> 1. 「공익신탁법」에 따른 공익신탁의 이익 2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득 가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득 나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다)

# 3. 답변생성

In [28]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [7]:
# from langchain_upstage import ChatOpenAI
# llm = ChatOpenAI(model='solar-pro2')

ImportError: cannot import name 'ChatOpenAI' from 'langchain_upstage' (C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\langchain_upstage\__init__.py)

In [29]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가이다. [context]를 참고해서 사용자 질문에 답하세요.[context]는 다음과 같아요
{retrieved_doc}
- 질문 :{query}"""

In [30]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉이 5천만 원인 직장인의 소득세 금액을 계산하려면, 종합소득세 과세 표준 계산과 세율 적용이 필요합니다. 아래 과정을 참고하세요.

1. 총급여액: 5,000만 원
2. 근로소득공제: 근로소득공제는 일정 표준공제액이 있으며, 소득구간별로 차등 적용됩니다.
   - 5,000만 원의 경우, 근로소득공제는 약 1,540만 원(대략적인 계산)입니다.
3. 과세표준 산출:
   - 과세표준 = 총급여액 - 근로소득공제
   - 대략: 5,000만 원 - 1,540만 원 = 약 3,460만 원
4. 소득세율 적용:
   - 3,460만 원의 경우, 소득세 과세표준 구간에 따라 세율이 차등 적용됩니다.
   - 일반적으로 3,460만 원은 24% 구간에 해당하는데, 2023 기준의 세율표를 참고하면
     - 1,200만 원 이하: 6%
     - 1,200만 원 초과 ~ 4,600만 원 이하: 15%~24%
     - 4,600만 원 초과: 35%
   - 따라서, 3,460만 원에 대해 적용되는 세율은 약 15%~24%입니다.

5. 세액 계산:
   - 대략적인 세액은 약 600만 원에서 700만 원 사이가 될 것으로 예상됩니다.
   - 세율 별 차등 계산을 고려하면, 약 640만 원 내외의 소득세를 납부할 가능성이 높습니다.

단, 정확한 세액은 개인의 세액공제 항목(연금보험료, 보험료공제, 인적공제 등), 기타 소득공제 여부에 따라 달라집니다. 구체적인 세액 계산을 위해서는 정확한 공제액과 공제 항목을 반영한 세무계산을 권장드립니다.

요약:  
연봉 5,000만 원인 직장인은 대략적으로 약 640만 원 내외의 소득세를 납부할 것으로 예상됩니다.  
정확한 금액 계산을 원하시면 구체적인 공제내역과 기타 소득구간을 알려주시면 상세히 안내드리겠습니다.


# 4.Langchain

In [31]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import ChatUpstage
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [33]:
# 1. 임배딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-large")
llm = ChatUpstage(model="solar-pro4")
# 2. vector DB load
pc = Pinecone()
index_name = 'tax-index-openai'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성
# (1) search_type'similarity':가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity',
                                    search_kwargs={'k':3})
# (2) search_type='mmr' : 후보10개중 서로 덜겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr',
#                                     search_kwargs={'k':3,
#                                                   'fetch_k':10,
#                                                   'lambda_mult':0.5}) # 1에 가까울수롯 관련성 깊은 계산
# (3) search_type='similarity_score_threshold':
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold',
#                                     search_kwargs={'k':3,'score_threshold':0.4})

# 4. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt = ChatPromptTemplate.from_template(template)

# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retriever_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# 6. RAG 채인 구성

rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)


제공된 문맥에는 구체적인 소득세율 구간이나 연말정산 공제 항목에 대한 정보가 없어 정확한 세액을 계산할 수 없습니다. 소득세액은 근로소득에서 근로소득공제, 인적공제 등 각종 공제를 차감한 과세표준에 누진세율을 적용하여 산정되며, 4대 보험료 및 연말정산 결과에 따라 달라집니다. 정확한 금액은 국세청 홈택스의 근로소득 간이세액표 또는 세무 전문가의 상담을 통해 확인하셔야 합니다.
